# 06 — Surrogate-Assisted Optimization (v2: CG + Controls)

Run the full optimization loop with **real CG** and **elevon trim** constraints:
1. Load trained 10-target surrogate (with Cl_beta, CLd01, Cmd01)
2. Configure mission + constraints (aero + propulsion + **trimmability**)
3. Surrogate-assisted Differential Evolution (with CG correction in reconstruct)
4. AVL validation of top candidates (with elevon trim)
5. Best feasible design with control authority check

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.aero.mission import MissionCondition, FeasibilityConfig
from src.surrogate.model import SurrogateModel
from src.optimization.problem import run_surrogate_assisted, run_differential_evolution
from src.optimization.database import EvaluationDatabase
from src.parameterization.design_variables import params_from_vector

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Configuration

In [ ]:
mission = MissionCondition()
feasibility = FeasibilityConfig()

print(f'Mission: V={mission.velocity} m/s, MTOW={mission.mtow} kg')
print(f'Mass budget: {mission.mass_budget:.3f} kg')
print(f'T_available @ cruise: {mission.thrust_available:.2f} N')
print()
print(f'v2 constraints:')
print(f'  SM range:            [{feasibility.sm_min:.0%}, {feasibility.sm_max:.0%}] MAC')
print(f'  Elevon defl max:     {feasibility.elevon_deflection_max:.0f} deg')
print(f'  Cl_beta/Cn_beta max: {feasibility.cl_beta_cn_beta_max:.0f}')
print(f'  T/D min:             {feasibility.td_min:.1f}')
print(f'  Endurance min:       {feasibility.endurance_min:.0f} s ({feasibility.endurance_min/60:.0f} min)')
print(f'  Vs max:              {feasibility.vs_max:.0f} m/s')
print(f'  Drag margin:         {feasibility.drag_margin:.2f}')

## 2. Run Surrogate-Assisted Optimization

In [ ]:
result = run_surrogate_assisted(
    mission=mission,
    feasibility=feasibility,
    surrogate_path='../models/surrogate_v2_ctrl',
    n_restarts=5,
    n_infill=50,
    max_cycles=8,
    surrogate_de_maxiter=300,
    surrogate_de_popsize=60,
    convergence_tol=0.05,
    seed=42,
    verbose=True,
)

## 3. Best Design Summary

In [ ]:
best = result['best_result']
best_params = result['best_params']

if best:
    print('=== Best Feasible Design (v2: real CG + elevon trim) ===')
    print(f'  L/D          = {best["L_over_D"]:.2f}')
    print(f'  CL           = {best["CL"]:.4f}')
    print(f'  CD           = {best["CD"]:.5f} (wing={best["CD0_wing"]:.5f} + body={best["CD0_body"]:.5f} + ind={best["CDi"]:.5f})')
    if 'CD_trim' in best:
        print(f'  CD_trim      = {best["CD_trim"]:.5f}')
    print(f'  CM           = {best["CM"]:.4f}')
    print(f'  Static margin= {best["static_margin"]:.3f} ({best["static_margin"]*100:.1f}% MAC)')
    print(f'  Cn_beta      = {best["Cn_beta"]:.4f}')
    if 'Cl_beta' in best:
        print(f'  Cl_beta      = {best["Cl_beta"]:.4f}')
    if 'elevon_deflection' in best:
        print(f'  Elevon defl  = {best["elevon_deflection"]:.1f} deg')
    if 'x_cg' in best:
        print(f'  CG x         = {best["x_cg"]*100:.1f} cm (x/c = {best["x_cg_frac"]:.3f})')
    print(f'  Span         = {2*best_params.half_span:.2f} m')
    print(f'  AR           = {best["AR"]:.1f}')
    print(f'  Struct mass  = {best["struct_mass"]:.3f} kg')
    print(f'  --- Propulsion ---')
    print(f'  T/D          = {best["T_over_D"]:.2f}')
    print(f'  Endurance    = {best["endurance_min"]:.1f} min')
    print(f'  Range        = {best["range_km"]:.1f} km')
    print(f'  Feasible     = {best["is_feasible"]}')
else:
    print('No feasible design found')
    print(f'Total AVL evals: {result["total_vlm_evals"]}')

## 4. Convergence

In [ ]:
if result.get('cycle_history'):
    cycles = result['cycle_history']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([c['cycle'] for c in cycles], [c['best_ld'] for c in cycles], 'bo-', lw=2, ms=8)
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Best Feasible L/D')
    ax.set_title('Surrogate-Assisted Optimization Convergence')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Total VLM evals: {result["total_vlm_evals"]}')

## 5. Save Results

In [ ]:
import json
from pathlib import Path

out = Path('../output')
out.mkdir(exist_ok=True)

if result.get('best_x') is not None:
    np.save(out / 'best_x_v2.npy', result['best_x'])
    print(f'Saved best_x_v2.npy')
    
if result.get('database'):
    result['database'].save(out / 'eval_database.json')
    
print(f'Results saved to {out}')